# 測試單一節點 — 在 Workflow 中測試 Prompt

**目標**:建立 LangGraph 節點、用 MLflow 追蹤、獨立測試

**前置條件**:
- 已完成 [01_hello_world.ipynb](01_hello_world.ipynb)
- 理解 LLMClient 基本用法


In [ ]:
from llm_framework.config import load_config
from llm_framework.llm_client import LLMClient
from llm_framework.workflow.state import WorkflowState, create_workflow_state
from llm_framework.mlflow.tracer import trace_node
from llm_framework.mlflow.experiment import ExperimentManager

load_config("dev")


## 什麼是 Workflow 節點?

節點是 workflow 中的一個處理步驟。每個節點接收 state,處理後回傳更新的 state。


In [ ]:
@trace_node("translator")
def translate_node(state: WorkflowState) -> dict:
    """翻譯節點:將中文翻譯成英文"""
    client = LLMClient()
    text = state["metadata"].get("text", "")
    messages = [
        {"role": "system", "content": "你是專業翻譯,將中文翻譯成英文。只回傳翻譯結果。"},
        {"role": "user", "content": text}
    ]
    response = client.chat(messages, temperature=0.0)
    return {"results": {"translation": response.content, "tokens": response.usage.total_tokens}}


## 測試節點


In [ ]:
manager = ExperimentManager()
with manager.start_run(run_name="test_translator"):
    state = create_workflow_state(metadata={"text": "今天天氣很好"})
    result = translate_node(state)
    print(f"翻譯結果: {result['results']['translation']}")
    print(f"Token 用量: {result['results']['tokens']}")


## 嘗試不同的 Prompt

修改 system prompt 來觀察效果差異。


In [ ]:
@trace_node("formal_translator")
def formal_translate(state: WorkflowState) -> dict:
    client = LLMClient()
    text = state["metadata"].get("text", "")
    messages = [
        {"role": "system", "content": "你是專業翻譯。將中文翻譯成正式的商業英文。"},
        {"role": "user", "content": text}
    ]
    response = client.chat(messages, temperature=0.0)
    return {"results": {"translation": response.content}}

with manager.start_run(run_name="test_formal_translator"):
    state = create_workflow_state(metadata={"text": "我們希望能盡快安排會議討論這個專案"})
    result = formal_translate(state)
    print(f"正式翻譯: {result['results']['translation']}")


## 總結

你已經學會了:
- 定義和測試 Workflow 節點
- 使用 @trace_node 裝飾器啟用 MLflow 追蹤
- 用 ExperimentManager 管理實驗

**下一步**:[03_full_workflow.ipynb](03_full_workflow.ipynb) — 完整 Workflow
